In [1]:
import pandas as pd

# pd.set_option('display.max_columns', None)  # none은 제한값 자리(몇개 열까지 보여줄까)
# pd.set_option('display.max_rows', None)

# 1-1. 데이터 불러오기
# 집
train = pd.read_parquet(r'C:\Users\soohok402\내 드라이브\Github\AI_Data_exhibition\parqueted_data\train_split')
val = pd.read_parquet(r'C:\Users\soohok402\내 드라이브\Github\AI_Data_exhibition\parqueted_data\val_split')

# 회사
# train = pd.read_parquet(r'C:\Users\20162180\AI_Data_exhibition\parqueted_data\train_split')
# val = pd.read_parquet(r'C:\Users\20162180\AI_Data_exhibition\parqueted_data\val_split')

from sklearn import set_config
set_config(display='text')
# scikit-learn의 HTML 출력 과정에서 발생한 cp949 간 UTF-8 충돌 때문에 넣는 코드

In [2]:
# 2-1 ablation 대상 컬럼군 설정
# 수위 컬럼군
wl_cols = ['wl_t',
          'wl_t_minus_1h',
          'wl_t_minus_3h',
          'wl_t_minus_6h',
          'wl_t_minus_12h',
          'wl_t_minus_24h',
          'target_maxrise_6h',  # 모델링 학습부터는 반드시 빼내야 함
]

# 강수 컬럼군
rain_cols = [
            # IMERG강우
            'imerg_rain_1h',
            'imerg_rain_6h',
            'imerg_rain_24h',
            # 지상강우
            'sfc_rain_1h',
            'sfc_rain_6h',
            'sfc_rain_24h',
            # 레이더 반사도
            'radar_reflectivity_mean',
            'radar_reflectivity_p95',
            'radar_reflectivity_ge20_fraction'
]

# 환경 컬럼군
env_cols = [
    'cat_area',       # km²
    'wsarea',         # km²
    'mean_elevation', # m
    'mean_slope',     # °
    'urban_ratio',    # %
    'forest_ratio',   # %
    'stream_order'
]

no_wl_cols = (train
              .select_dtypes(include = 'number')
              .columns
              .drop(wl_cols)  # 종속변수 여기서 제거됨
)

no_rain_cols = (train
                .select_dtypes(include = 'number')
                .columns
                .drop(rain_cols + ['target_maxrise_6h'])  # 종속변수 제거
)

no_env_cols = (train
               .select_dtypes(include = 'number')
               .columns
               .drop(env_cols + ['target_maxrise_6h'])  # 종속변수 제거
)


datasets_ablation = {
    'no_wl_cols' : no_wl_cols,
    'no_rain_cols' : no_rain_cols,
    'no_env_cols' : no_env_cols
}

print(datasets_ablation.items())  # datasets의 이름과 내용물이 같이 나옴(반환값 두개)

dict_items([('no_wl_cols', Index(['imerg_rain_1h', 'imerg_rain_6h', 'imerg_rain_24h', 'sfc_rain_1h',
       'sfc_rain_6h', 'sfc_rain_24h', 'radar_reflectivity_mean',
       'radar_reflectivity_p95', 'radar_reflectivity_ge20_fraction',
       'cat_area', 'wsarea', 'mean_elevation', 'mean_slope', 'urban_ratio',
       'forest_ratio', 'stream_order', 'event_id'],
      dtype='object')), ('no_rain_cols', Index(['wl_t', 'wl_t_minus_1h', 'wl_t_minus_3h', 'wl_t_minus_6h',
       'wl_t_minus_12h', 'wl_t_minus_24h', 'cat_area', 'wsarea',
       'mean_elevation', 'mean_slope', 'urban_ratio', 'forest_ratio',
       'stream_order', 'event_id'],
      dtype='object')), ('no_env_cols', Index(['wl_t', 'wl_t_minus_1h', 'wl_t_minus_3h', 'wl_t_minus_6h',
       'wl_t_minus_12h', 'wl_t_minus_24h', 'imerg_rain_1h', 'imerg_rain_6h',
       'imerg_rain_24h', 'sfc_rain_1h', 'sfc_rain_6h', 'sfc_rain_24h',
       'radar_reflectivity_mean', 'radar_reflectivity_p95',
       'radar_reflectivity_ge20_fraction', 'e

In [3]:
# 2-2 종속변수 설정
y_train = train['target_maxrise_6h']
y_val = val['target_maxrise_6h']

In [4]:
# 2-3. OHE station_id OHE 포함 모델링 데이터셋
# 포함한게 조금이나마 나은 결과, 포함해서 모델링 시행

# 원핫인코딩

from sklearn.preprocessing import OneHotEncoder

# 원핫인코더 생성
ohe = OneHotEncoder(
    handle_unknown='ignore',  # train에 없던 범주가 나와도 에러 X
    sparse_output=True  # sparse: 희소한, 희소행렬(0이 아닌 값이 드문 행렬)
    # 원핫인코딩 결과를 전체 다 저장 X, 희소한 1의 값의 위치만 저장(메모리 사용 적음)
)

# 학습은 train만
ohe.fit(train[['station_id']])  # sklearn 인코더는 2차원 행렬을 받는다.

# 변환
train_st_id_ohe = ohe.transform(train[['station_id']])
val_st_id_ohe = ohe.transform(val[['station_id']])



In [5]:
# 3. 모델링
# ohe와 합치기~모델링~평가 및 검증까지 for 문안에 한번에 만들기

# 3-1. 변수군별 제외 모델링

from scipy.sparse import hstack
from lightgbm import LGBMRegressor
from sklearn.metrics import root_mean_squared_error

# 해당 데이터셋 ohe와 합치기
result = {}  # 조건별 rmse 저장용 딕셔너리

for name, cols in datasets_ablation.items(): # name: 변수군의 이름, cols: 컬럼군
    print(name)
    X_train = hstack([
        train[cols],
        train_st_id_ohe
    ]).tocsr()
    
    X_val = hstack([
        val[cols],
        val_st_id_ohe
    ]).tocsr()

    # 모델링
    model = LGBMRegressor(
        random_state = 42
    )

    model.fit(X_train, y_train)

    pred = model.predict(X_val)

    # 성능평가
    rmse = root_mean_squared_error(y_val, pred)

    # 결과에 더하기
    result[name] = round(rmse, 4)  # result는 딕셔너리, []는 key가 됨
    print()

print(result)

no_wl_cols
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.015500 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 5029
[LightGBM] [Info] Number of data points in the train set: 616331, number of used features: 482
[LightGBM] [Info] Start training from score 0.111701

no_rain_cols
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.015669 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 4301
[LightGBM] [Info] Number of data points in the train set: 616331, number of used features: 479
[LightGBM] [Info] Start training from score 0.111701

no_env_cols
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.013669 seconds.
You can set `force_row_wise=true` to remov

In [6]:
# 3-2. 수위 변수별 모델링: 몇시간 전의 수위가 가장 영향이 큰가?
num_cols = (train
            .select_dtypes(include = 'number')
            .columns
            .drop(['target_maxrise_6h', 'event_id'])
)

wl_var_ablation = {
    'no_wl_t' : 'wl_t',
    'no_wl_t_minus_1h' : 'wl_t_minus_1h',
    'no_wl_t_minus_3h' : 'wl_t_minus_3h',
    'no_wl_t_minus_6h' : 'wl_t_minus_6h',
    'no_wl_t_minus_12h' : 'wl_t_minus_12h',
    'no_wl_t_minus_24h' : 'wl_t_minus_24h',
}


for name, col in wl_var_ablation.items(): # name: 변수군의 이름, cols: 컬럼군
    print(name)
    X_train = hstack([
        train[num_cols.drop(col)],
        train_st_id_ohe
    ]).tocsr()
    
    X_val = hstack([
        val[num_cols.drop(col)],
        val_st_id_ohe
    ]).tocsr()

    # 모델링
    model = LGBMRegressor(
        random_state = 42
    )

    model.fit(X_train, y_train)

    pred = model.predict(X_val)

    # 성능평가
    rmse = root_mean_squared_error(y_val, pred)

    # 결과에 더하기
    result[name] = round(rmse, 4)  # result는 딕셔너리, []는 key가 됨

print(result)

no_wl_t
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.058912 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 6205
[LightGBM] [Info] Number of data points in the train set: 616331, number of used features: 486
[LightGBM] [Info] Start training from score 0.111701
no_wl_t_minus_1h
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.053619 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 6205
[LightGBM] [Info] Number of data points in the train set: 616331, number of used features: 486
[LightGBM] [Info] Start training from score 0.111701
no_wl_t_minus_3h
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.043290 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 6205
[LightGBM] [Info] Number of data points in the train set: 616331, number o

In [ ]:
# 3-2(추가). 수위변수군 영향력 >> 개별변수군의 합  -> 수위변수군 변수간 상호작용 확인: 상관계수
print("피어슨상관계수: 수위변수군")
display(train[wl_cols].corr(method = 'pearson'))
print()
print("스피어만상관계수: 수위변수군")
display(train[wl_cols].corr(method = 'spearman'))

# 정리 필요
# 수위변수군 내부적으로 강력한 상관성이 확인됨
# 그러나 개별 변수의 종속변수에 대한 상관성은 미미함
# -> 개별값보다 시차별 수위 변화에 대한 정보가 예측에 활용됐을 가능성
# -> 차분에 의한 파생변수 생성 검토 필요


피어슨상관계수: 수위변수군


,wl_t,wl_t_minus_1h,wl_t_minus_3h,wl_t_minus_6h,wl_t_minus_12h,wl_t_minus_24h,target_maxrise_6h
wl_t,1.000000,0.999956,0.999734,0.999459,0.999625,0.999451,-0.031109
wl_t_minus_1h,0.999956,1.000000,0.999858,0.999531,0.999627,0.999436,-0.032974
wl_t_minus_3h,0.999734,0.999858,1.000000,0.999750,0.999554,0.999349,-0.030450
wl_t_minus_6h,0.999459,0.999531,0.999750,1.000000,0.999529,0.999341,-0.021632
wl_t_minus_12h,0.999625,0.999627,0.999554,0.999529,1.000000,0.999799,-0.033898
wl_t_minus_24h,0.999451,0.999436,0.999349,0.999341,0.999799,1.000000,-0.033684
target_maxrise_6h,-0.031109,-0.032974,-0.030450,-0.021632,-0.033898,-0.033684,1.000000



스피어만상관계수: 수위변수군


,wl_t,wl_t_minus_1h,wl_t_minus_3h,wl_t_minus_6h,wl_t_minus_12h,wl_t_minus_24h,target_maxrise_6h
wl_t,1.000000,0.995125,0.976582,0.955283,0.961340,0.942837,-0.007705
wl_t_minus_1h,0.995125,1.000000,0.986422,0.961260,0.961352,0.941398,-0.026904
wl_t_minus_3h,0.976582,0.986422,1.000000,0.977823,0.958195,0.937366,-0.030742
wl_t_minus_6h,0.955283,0.961260,0.977823,1.000000,0.960984,0.940194,-0.008994
wl_t_minus_12h,0.961340,0.961352,0.958195,0.960984,1.000000,0.976016,-0.034620
wl_t_minus_24h,0.942837,0.941398,0.937366,0.940194,0.976016,1.000000,-0.029790
target_maxrise_6h,-0.007705,-0.026904,-0.030742,-0.008994,-0.034620,-0.029790,1.000000


In [ ]:
# 3-3. 강우변수 세분화해서 영향력 확인
rain_vars_ablation = {
    'no_imerg' : [
            'imerg_rain_1h',
            'imerg_rain_6h',
            'imerg_rain_24h'
    ],
    'no_sfc' : [
            'sfc_rain_1h',
            'sfc_rain_6h',
            'sfc_rain_24h'
    ],
    'no_radar_reflect' : [
            'radar_reflectivity_mean',
            'radar_reflectivity_p95',
            'radar_reflectivity_ge20_fraction'
    ]
}

for name, cols in rain_vars_ablation.items(): # name: 변수군의 이름, cols: 컬럼군
    print(name)
    X_train = hstack([
        train[num_cols.drop(cols)],
        train_st_id_ohe
    ]).tocsr()
    
    X_val = hstack([
        val[num_cols.drop(cols)],
        val_st_id_ohe
    ]).tocsr()

    # 모델링
    model = LGBMRegressor(
        random_state = 42
    )

    model.fit(X_train, y_train)

    pred = model.predict(X_val)

    # 성능평가
    rmse = root_mean_squared_error(y_val, pred)

    # 결과에 더하기
    result[name] = round(rmse, 4)  # result는 딕셔너리, []는 key가 됨

no_imerg
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.014229 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 5695
[LightGBM] [Info] Number of data points in the train set: 616331, number of used features: 484
[LightGBM] [Info] Start training from score 0.111701
no_sfc
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.060322 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 5727
[LightGBM] [Info] Number of data points in the train set: 616331, number of used features: 484
[LightGBM] [Info] Start training from score 0.111701
no_radar_reflect
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.055979 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 5700
[LightGBM] [Info] N

In [ ]:
# rmse 결과 df로 보기

result_df = pd.DataFrame(
result.items(),
columns=['model', 'rmse']
)

result_df['from_base_model'] = round((result_df['rmse'] - 0.2133), 4)

result_df

,model,rmse,from_base_model
0,no_wl_cols,0.3169,0.1036
1,no_rain_cols,0.2383,0.0250
2,no_env_cols,0.2270,0.0137
3,no_wl_t,0.2142,0.0009
4,no_wl_t_minus_1h,0.2161,0.0028
5,no_wl_t_minus_3h,0.2153,0.0020
6,no_wl_t_minus_6h,0.2244,0.0111
7,no_wl_t_minus_12h,0.2133,0.0000
8,no_wl_t_minus_24h,0.2136,0.0003
9,no_imerg,0.2144,0.0011
